In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the train dataset
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\ml_benchmark\\04_titanic\\split_train.csv'
train_df = pd.read_csv(train_data_path)

# Display the first few rows of the dataset
print(train_df.head())

# Check the data types and basic information
print(train_df.info())

# Check for missing values
print(train_df.isnull().sum())

# Summary statistics for numerical columns
print(train_df.describe())

# Summary statistics for categorical columns
print(train_df.describe(include=['O']))

# Distribution of the target variable
sns.countplot(x='Survived', data=train_df)
plt.title('Distribution of Survival')
plt.show()

# Distribution of numerical features
numerical_features = train_df.select_dtypes(include=[np.number]).columns.tolist()
for feature in numerical_features:
    plt.figure()
    sns.histplot(train_df[feature], kde=True)
    plt.title(f'Distribution of {feature}')
    plt.show()

# Distribution of categorical features
categorical_features = train_df.select_dtypes(include=['O']).columns.tolist()
for feature in categorical_features:
    plt.figure()
    sns.countplot(x=feature, data=train_df)
    plt.title(f'Distribution of {feature}')
    plt.xticks(rotation=45)
    plt.show()

# Correlation matrix for numerical features
correlation_matrix = train_df[numerical_features].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()


   PassengerId  Survived  Pclass  ...      Fare Cabin  Embarked
0          409         0       3  ...    7.7750   NaN         S
1          481         0       3  ...   46.9000   NaN         S
2          511         1       3  ...    7.7500   NaN         Q
3          610         1       1  ...  153.4625  C125         S
4          548         1       2  ...   13.8625   NaN         C

[5 rows x 12 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Survived     712 non-null    int64  
 2   Pclass       712 non-null    int64  
 3   Name         712 non-null    object 
 4   Sex          712 non-null    object 
 5   Age          567 non-null    float64
 6   SibSp        712 non-null    int64  
 7   Parch        712 non-null    int64  
 8   Ticket       712 non-null    object 
 9   Fare         712 non-

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-08-30 15:58:04.296 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], 'Numeric': ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale
from metagpt.tools.libs.feature_engineering import TargetMeanEncoder
import pandas as pd

# Load the eval data
eval_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\ml_benchmark\\04_titanic\\split_eval.csv'
eval_df = pd.read_csv(eval_data_path)

# Copy the dataframes to avoid modifying the original data
train_df_copy = train_df.copy()
eval_df_copy = eval_df.copy()

# Handle missing values
# For numerical features, use the median
num_features = ['Age', 'Fare']
fill_missing_num = FillMissingValue(num_features, strategy='median')
train_df_copy = fill_missing_num.fit_transform(train_df_copy)
eval_df_copy = fill_missing_num.transform(eval_df_copy)

# For categorical features, use the most frequent value
cat_features = ['Embarked', 'Cabin']
fill_missing_cat = FillMissingValue(cat_features, strategy='most_frequent')
train_df_copy = fill_missing_cat.fit_transform(train_df_copy)
eval_df_copy = fill_missing_cat.transform(eval_df_copy)

# Encode categorical variables
# Use target mean encoding for 'Embarked' and 'Cabin'
target_mean_encoder_embarked = TargetMeanEncoder('Embarked', 'Survived')
train_df_copy = target_mean_encoder_embarked.fit_transform(train_df_copy)
eval_df_copy = target_mean_encoder_embarked.transform(eval_df_copy)

target_mean_encoder_cabin = TargetMeanEncoder('Cabin', 'Survived')
train_df_copy = target_mean_encoder_cabin.fit_transform(train_df_copy)
eval_df_copy = target_mean_encoder_cabin.transform(eval_df_copy)

# For 'Sex', we can use a simple mapping
train_df_copy['Sex'] = train_df_copy['Sex'].map({'male': 0, 'female': 1})
eval_df_copy['Sex'] = eval_df_copy['Sex'].map({'male': 0, 'female': 1})

# Normalize numerical features
# Exclude 'PassengerId' and 'Survived' from scaling
num_features_to_scale = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
scaler = StandardScale(num_features_to_scale)
train_df_copy = scaler.fit_transform(train_df_copy)
eval_df_copy = scaler.transform(eval_df_copy)

# Display the preprocessed data
print(train_df_copy.head())
print(eval_df_copy.head())


   PassengerId  Survived  ...  Embarked_target_mean Cabin_target_mean
0          409         0  ...              0.331407          0.291513
1          481         0  ...              0.331407          0.291513
2          511         1  ...              0.362069          0.291513
3          610         1  ...              0.331407          1.000000
4          548         1  ...              0.548148          0.291513

[5 rows x 14 columns]
   PassengerId  Survived  ...  Embarked_target_mean Cabin_target_mean
0          206         0  ...              0.331407          0.500000
1           45         1  ...              0.362069          0.291513
2          822         1  ...              0.331407          0.291513
3          459         1  ...              0.331407          0.291513
4          796         0  ...              0.331407          0.291513

[5 rows x 14 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': ['Name', 'Ticket', 'Cabin', 'Embarked'], 'Numeric': ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_target_mean', 'Cabin_target_mean'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from metagpt.tools.libs.data_preprocess import LabelEncode

# Label encode non-numeric columns
non_numeric_features = ['Name', 'Ticket', 'Cabin', 'Embarked']
label_encoder = LabelEncode(non_numeric_features)
train_df_copy = label_encoder.fit_transform(train_df_copy)
eval_df_copy = label_encoder.transform(eval_df_copy)

# Prepare the data for training
X_train = train_df_copy.drop(['Survived', 'PassengerId'], axis=1)
y_train = train_df_copy['Survived']
X_eval = eval_df_copy.drop(['Survived', 'PassengerId'], axis=1)
y_eval = eval_df_copy['Survived']

# Define the model
model = RandomForestClassifier(random_state=42)

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Evaluate the model on the evaluation set
y_pred = best_model.predict(X_eval)
accuracy = accuracy_score(y_eval, y_pred)
print(f'Accuracy on eval data: {accuracy:.4f}')


ValueError: Input X contains NaN.
RandomForestClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from metagpt.tools.libs.data_preprocess import LabelEncode

# Label encode non-numeric columns
non_numeric_features = ['Name', 'Ticket', 'Cabin', 'Embarked']
label_encoder = LabelEncode(non_numeric_features)
train_df_copy = label_encoder.fit_transform(train_df_copy)
eval_df_copy = label_encoder.transform(eval_df_copy)

# Impute missing values
numeric_features = ['Age', 'Fare', 'Embarked_target_mean', 'Cabin_target_mean']
categorical_features = ['Pclass', 'Sex', 'SibSp', 'Parch']

numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

train_df_copy[numeric_features] = numeric_imputer.fit_transform(train_df_copy[numeric_features])
eval_df_copy[numeric_features] = numeric_imputer.transform(eval_df_copy[numeric_features])

train_df_copy[categorical_features] = categorical_imputer.fit_transform(train_df_copy[categorical_features])
eval_df_copy[categorical_features] = categorical_imputer.transform(eval_df_copy[categorical_features])

# Prepare the data for training
X_train = train_df_copy.drop(['Survived', 'PassengerId'], axis=1)
y_train = train_df_copy['Survived']
X_eval = eval_df_copy.drop(['Survived', 'PassengerId'], axis=1)
y_eval = eval_df_copy['Survived']

# Define the model
model = RandomForestClassifier(random_state=42)

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Evaluate the model on the evaluation set
y_pred = best_model.predict(X_eval)
accuracy = accuracy_score(y_eval, y_pred)
print(f'Accuracy on eval data: {accuracy:.4f}')

Accuracy on eval data: 0.7933
